<a href="https://colab.research.google.com/github/armandoordonez/AI-Engineering/blob/main/tutorial_langchain_langsmith.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial práctico: LangChain + LangSmith

**LangChain** sirve para construir aplicaciones con LLMs (prompts, cadenas, herramientas, agentes). **LangSmith** sirve para observarlas, depurarlas y evaluarlas.

**Contenido**
1. Instalación
2. Claves y configuración de LangSmith
3. Primera llamada al modelo
4. Prompts y cadenas (LCEL)
5. Salida estructurada
6. Herramientas y agentes
7. RAG básico
8. `@traceable`: trazar tu propio código
9. Evaluación con datasets
10. Memoria conversacional

> Requiere: cuenta en [smith.langchain.com](https://smith.langchain.com) y una API key de Anthropic **u** OpenAI. Usa la API de LangChain 1.x.

## 1. Instalación

In [1]:
%pip install -qU langchain langchain-anthropic langchain-openai langsmith langgraph

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.0 MB/s eta 0:00:00


## 2. Claves y configuración de LangSmith

Con `LANGSMITH_TRACING=true`, todo lo que ejecutes con LangChain queda registrado en LangSmith automáticamente.
Las claves se piden con `getpass` para que no queden guardadas en el notebook.

In [3]:
import os
from google.colab import userdata

def set_key_from_secrets(name):
    try:
        os.environ[name] = userdata.get(name)
    except Exception as e:
        print(f"No se pudo cargar {name} desde los secretos de Colab: {e}")

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "tutorial-langchain"

# Cargar LangSmith
set_key_from_secrets("LANGSMITH_API_KEY")

# Elige tu proveedor: "anthropic" u "openai"
PROVEEDOR = "anthropic"
key_name = "ANTHROPIC_API_KEY" if PROVEEDOR == "anthropic" else "OPENAI_API_KEY"
set_key_from_secrets(key_name)

## 3. Primera llamada al modelo

In [4]:
from langchain.chat_models import init_chat_model

MODELO = "anthropic:claude-sonnet-4-5" if PROVEEDOR == "anthropic" else "openai:gpt-4o-mini"
model = init_chat_model(MODELO, temperature=0)

resp = model.invoke("Explica qué es un LLM en una frase.")
print(resp.content)
print(resp.usage_metadata)  # tokens usados

Un LLM (Large Language Model) es un modelo de inteligencia artificial entrenado con enormes cantidades de texto para comprender y generar lenguaje humano de manera coherente.
{'input_tokens': 22, 'output_tokens': 46, 'total_tokens': 68, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}}


👉 Abre LangSmith → proyecto **tutorial-langchain**. Ahí verás esta llamada con su input, output, tokens y latencia.

## 4. Prompts y cadenas (LCEL)
El operador `|` encadena componentes: **prompt → modelo → parser**.

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un profesor experto en {tema}. Responde en español, en máximo 3 frases."),
    ("human", "{pregunta}"),
])

chain = prompt | model | StrOutputParser()
print(chain.invoke({"tema": "finanzas", "pregunta": "¿Qué es el interés compuesto?"}))

El interés compuesto es el interés que se calcula sobre el capital inicial más los intereses acumulados de períodos anteriores, generando un efecto de "interés sobre interés". A diferencia del interés simple, permite que tu dinero crezca de forma exponencial con el tiempo. Es una de las herramientas más poderosas para la creación de riqueza a largo plazo.


In [6]:
# Varias entradas en paralelo
respuestas = chain.batch([
    {"tema": "física", "pregunta": "¿Qué es la entropía?"},
    {"tema": "historia", "pregunta": "¿Qué fue la Ilustración?"},
])
for r in respuestas:
    print("-", r, "\n")

- La entropía es una medida del desorden o la dispersión de la energía en un sistema físico. Según la segunda ley de la termodinámica, la entropía del universo tiende siempre a aumentar, lo que significa que los procesos naturales evolucionan hacia estados más desordenados. En términos prácticos, representa la cantidad de energía que ya no está disponible para realizar trabajo útil. 

- La Ilustración fue un movimiento intelectual y cultural del siglo XVIII que enfatizó la razón, la ciencia y el pensamiento crítico como medios para reformar la sociedad y avanzar el conocimiento. Surgió principalmente en Europa, con pensadores como Voltaire, Rousseau, Montesquieu y Kant, quienes cuestionaron las tradiciones, el absolutismo y los dogmas religiosos. Este movimiento sentó las bases ideológicas para revoluciones políticas como la francesa y la independencia americana. 



In [7]:
# Streaming token a token
for chunk in chain.stream({"tema": "IA", "pregunta": "¿Qué es RAG?"}):
    print(chunk, end="", flush=True)

RAG (Retrieval-Augmented Generation) es una técnica de IA que combina la recuperación de información de una base de datos o documentos externos con un modelo de lenguaje generativo. Permite que los modelos accedan a conocimiento actualizado y específico sin necesidad de reentrenamiento, mejorando la precisión y reduciendo las alucinaciones. Es especialmente útil para crear chatbots y asistentes que necesitan información actualizada o datos empresariales específicos.

## 5. Salida estructurada
Pide al modelo un objeto Pydantic en lugar de texto libre.

In [9]:
from pydantic import BaseModel, Field

class Resumen(BaseModel):
    titulo: str
    puntos_clave: list[str] = Field(description="3 a 5 ideas principales")
    sentimiento: str = Field(description="positivo, negativo o neutro")

extractor = model.with_structured_output(Resumen)

texto = '''La empresa reportó ventas récord en el tercer trimestre, impulsadas por
el canal digital. Sin embargo, los costos logísticos subieron 12% y la gerencia
anunció un plan de eficiencia para 2027.'''

r = extractor.invoke(f"Resume este texto:\n{texto}")
r

Resumen(titulo='Ventas récord en Q3 con desafíos en costos logísticos', puntos_clave=['La empresa alcanzó ventas récord en el tercer trimestre gracias al canal digital', 'Los costos logísticos aumentaron un 12%', 'La gerencia anunció un plan de eficiencia para implementar en 2027'], sentimiento='neutro')

## 6. Herramientas y agentes
El agente decide cuándo llamar cada herramienta.

In [10]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool
def clima(ciudad: str) -> str:
    """Devuelve el clima actual de una ciudad."""
    return f"En {ciudad} hay 24°C y está soleado."  # aquí llamarías una API real

@tool
def multiplicar(a: float, b: float) -> float:
    """Multiplica dos números."""
    return a * b

agent = create_agent(
    model=model,
    tools=[clima, multiplicar],
    system_prompt="Eres un asistente útil. Usa herramientas cuando las necesites. Responde en español.",
)

result = agent.invoke({"messages": [
    {"role": "user", "content": "¿Qué clima hace en Cali? Y ¿cuánto es 23.5 por 4?"}
]})

for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

¿Qué clima hace en Cali? Y ¿cuánto es 23.5 por 4?
================================== Ai Message ==================================

[{'id': 'toolu_01PaScRGk316r8XzUa8f7ELT', 'caller': {'type': 'direct'}, 'input': {'ciudad': 'Cali'}, 'name': 'clima', 'type': 'tool_use', 'toolset_name': None}, {'id': 'toolu_01NUWNLja5cdDqKzyueoa5de', 'caller': {'type': 'direct'}, 'input': {'a': 23.5, 'b': 4}, 'name': 'multiplicar', 'type': 'tool_use', 'toolset_name': None}]
Tool Calls:
  clima (toolu_01PaScRGk316r8XzUa8f7ELT)
 Call ID: toolu_01PaScRGk316r8XzUa8f7ELT
  Args:
    ciudad: Cali
  multiplicar (toolu_01NUWNLja5cdDqKzyueoa5de)
 Call ID: toolu_01NUWNLja5cdDqKzyueoa5de
  Args:
    a: 23.5
    b: 4
================================= Tool Message =================================
Name: clima

En Cali hay 24°C y está soleado.
================================= Tool Message =================================
Name: multipli

👉 En LangSmith verás el árbol del agente: modelo → herramienta(s) → modelo. Esto ayuda mucho a depurar.

## 7. RAG básico
Responder con base en tus propios documentos. Para no depender de otra API key, aquí se usa un buscador **BM25** (por palabras clave).
En producción normalmente usarías embeddings y una base vectorial (Chroma, FAISS, pgvector...).

In [13]:
%pip install -qU langchain-community rank_bm25

In [14]:
from langchain_community.retrievers import BM25Retriever

politicas = [
    "Vacaciones: cada empleado tiene 15 días hábiles de vacaciones al año.",
    "Trabajo remoto: se permite trabajo remoto 2 días por semana, previa aprobación del jefe.",
    "Reembolsos: los reembolsos de gastos se solicitan antes del día 25 de cada mes.",
    "Capacitación: la empresa financia hasta el 70% de cursos relacionados con el cargo.",
]
retriever = BM25Retriever.from_texts(politicas, k=2)

@tool
def buscar_politicas(consulta: str) -> str:
    """Busca en las políticas internas de la empresa."""
    return "\n".join(d.page_content for d in retriever.invoke(consulta))

rag_agent = create_agent(
    model,
    tools=[buscar_politicas],
    system_prompt="Responde SOLO con base en las políticas encontradas. Si no está, di que no sabes.",
)

out = rag_agent.invoke({"messages": [{"role": "user", "content": "¿Cuántos días de remoto puedo hacer?"}]})
print(out["messages"][-1].content)

Según las políticas de la empresa, puedes hacer **2 días de trabajo remoto por semana**, previa aprobación de tu jefe.


**Con PDFs reales** (opcional):
```python
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
docs = PyPDFLoader("mi_documento.pdf").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100).split_documents(docs)
retriever = BM25Retriever.from_documents(chunks, k=4)
```

## 8. `@traceable`: trazar tu propio código
Así puedes rastrear en LangSmith funciones que no son de LangChain.

In [15]:
from langsmith import traceable

@traceable
def clasificar(texto: str) -> str:
    return "facturación" if "factura" in texto.lower() else "atención general"

@traceable(name="pipeline_soporte", tags=["tutorial"], metadata={"version": "v1"})
def responder_ticket(texto: str) -> str:
    categoria = clasificar(texto)
    return chain.invoke({"tema": categoria, "pregunta": texto})

print(responder_ticket("No me llegó la factura de septiembre, ¿qué hago?"))

Te recomiendo que sigas estos pasos:

1. **Revisa tu carpeta de spam o correo no deseado**, ya que a veces las facturas pueden llegar allí por error.

2. **Contacta directamente con el departamento de facturación** de tu proveedor para solicitar que te reenvíen la factura de septiembre o accede a tu área de cliente online donde suelen estar disponibles para descargar.

3. Si necesitas la factura con urgencia, solicita una copia por correo electrónico o descárgala desde la plataforma web del servicio.


## 9. Evaluación con datasets
1. Creas un dataset con entradas y salidas esperadas.
2. Defines la función a evaluar (*target*) y uno o más evaluadores.
3. Corres el experimento y lo comparas en la interfaz de LangSmith.

In [17]:
from langsmith import Client

client = Client()
DATASET = "preguntas-basicas-tutorial"

if not client.has_dataset(dataset_name=DATASET):
    ds = client.create_dataset(DATASET, description="Dataset de prueba del tutorial")
    client.create_examples(dataset_id=ds.id, examples=[
        {"inputs": {"pregunta": "¿Cuál es la capital de Colombia?"}, "outputs": {"respuesta": "Bogotá"}},
        {"inputs": {"pregunta": "¿Cuánto es 2 + 2?"},                "outputs": {"respuesta": "4"}},
        {"inputs": {"pregunta": "¿En qué continente está Egipto?"},  "outputs": {"respuesta": "África"}},
    ])

In [18]:
qa = ChatPromptTemplate.from_template("Responde solo con la respuesta, sin explicación: {pregunta}") | model | StrOutputParser()

def target(inputs: dict) -> dict:
    return {"respuesta": qa.invoke(inputs)}

# Evaluador 1: coincidencia simple
def contiene_respuesta(outputs: dict, reference_outputs: dict) -> bool:
    return reference_outputs["respuesta"].lower() in outputs["respuesta"].lower()

# Evaluador 2: brevedad
def es_breve(outputs: dict) -> bool:
    return len(outputs["respuesta"].split()) <= 5

resultados = client.evaluate(
    target,
    data=DATASET,
    evaluators=[contiene_respuesta, es_breve],
    experiment_prefix="v1",
)
resultados.to_pandas()

View the evaluation results for experiment: 'v1-3b84d8c5' at:
https://smith.langchain.com/o/d442167e-d5fc-5d8b-a440-bd8146a0707a/datasets/325b7b83-06b7-4c2b-a749-e36a1fa9ff0f/compare?selectedSessions=5a80666b-d075-49e9-8014-6781b291c9bb




0it [00:00, ?it/s]

,inputs.pregunta,outputs.respuesta,error,reference.respuesta,feedback.contiene_respuesta,feedback.es_breve,execution_time,example_id,id
0,¿En qué continente está Egipto?,África,None,África,True,True,1.288342,01777232-1bb7-49a1-b419-ca2fdef9f56f,01a1559e-07bb-7752-8167-228bb9d85b68
1,¿Cuál es la capital de Colombia?,Bogotá,None,Bogotá,True,True,1.066632,2846bb2b-b29f-47d0-bb2e-acdb7d91a352,01a1559e-0cc9-7042-8aaf-17d287849050
2,¿Cuánto es 2 + 2?,4,None,4,True,True,1.124759,94b365ff-3bd9-4734-9944-b6cbc18d6281,01a1559e-10f6-75f2-9982-cb95c03786c7


### LLM como juez (para respuestas abiertas)

In [19]:
class Veredicto(BaseModel):
    correcto: bool
    razon: str

juez = model.with_structured_output(Veredicto)

def llm_juez(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    v = juez.invoke(
        f"Pregunta: {inputs['pregunta']}\n"
        f"Respuesta de referencia: {reference_outputs['respuesta']}\n"
        f"Respuesta del modelo: {outputs['respuesta']}\n"
        "¿La respuesta del modelo es correcta respecto a la referencia?"
    )
    return {"key": "llm_juez", "score": v.correcto, "comment": v.razon}

client.evaluate(target, data=DATASET, evaluators=[llm_juez], experiment_prefix="v1-juez")

View the evaluation results for experiment: 'v1-juez-8a27e7b0' at:
https://smith.langchain.com/o/d442167e-d5fc-5d8b-a440-bd8146a0707a/datasets/325b7b83-06b7-4c2b-a749-e36a1fa9ff0f/compare?selectedSessions=75569aab-8ffd-435d-8698-db07b8470998




0it [00:00, ?it/s]

,inputs.pregunta,outputs.respuesta,error,reference.respuesta,feedback.llm_juez,execution_time,example_id,id
0,¿En qué continente está Egipto?,África,None,África,True,1.179123,01777232-1bb7-49a1-b419-ca2fdef9f56f,01a1559e-3f73-7b90-a0bc-b38514b548bd
1,¿Cuál es la capital de Colombia?,Bogotá,None,Bogotá,True,1.148928,2846bb2b-b29f-47d0-bb2e-acdb7d91a352,01a1559e-4b82-7c83-b466-0c38cdc6552c
2,¿Cuánto es 2 + 2?,4,None,4,True,2.087607,94b365ff-3bd9-4734-9944-b6cbc18d6281,01a1559e-57f3-7b31-98e2-f159bdcb5db7


👉 En LangSmith → **Datasets & Experiments** puedes comparar los experimentos lado a lado cuando cambies el prompt o el modelo.

## 10. Memoria conversacional
Con un *checkpointer*, el agente recuerda la conversación de cada `thread_id`.

In [20]:
from langgraph.checkpoint.memory import InMemorySaver

agente_memoria = create_agent(model, tools=[clima], checkpointer=InMemorySaver(),
                              system_prompt="Eres un asistente amable. Responde en español.")
config = {"configurable": {"thread_id": "jose-1"}}

agente_memoria.invoke({"messages": [{"role": "user", "content": "Hola, me llamo José y vivo en Cali."}]}, config)
r = agente_memoria.invoke({"messages": [{"role": "user", "content": "¿Cómo me llamo y qué clima hace donde vivo?"}]}, config)
print(r["messages"][-1].content)

¡Perfecto! En Cali actualmente hay **24°C y está soleado**. ¡Un clima muy agradable! 😊


## Próximos pasos
- **Prompt Hub**: versiona prompts con `client.push_prompt("mi-prompt", object=prompt)` y recupéralos con `client.pull_prompt("mi-prompt")`.
- **LangGraph**: para flujos con varios pasos, aprobación humana o varios agentes.
- **Producción**: monitoreo, alertas y *online evaluators* en LangSmith sobre el tráfico real.
- Documentación: [docs.langchain.com](https://docs.langchain.com)